# Custom Segmentation Training — RF-DETR Seg

Train an instance-segmentation model (RF-DETR Seg): each object's box and outline on your Roboflow dataset (COCO format) and download a `.pkl`
for the Jetson vision server.

The `.pkl` carries the model as **ONNX**. On the Jetson, run `model_prepare`
with it once: that builds its TensorRT engine for the Jetson's GPU (a few
minutes; an engine is tied to the GPU that builds it, so it cannot be built here).

**Important:** Set **Runtime > Change runtime type > GPU** before starting.

## 1. Download Dataset from Roboflow

Export your Roboflow project as **Instance Segmentation**.

In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="YOUR_API_KEY_HERE")
project = rf.workspace("your-workspace").project("your-project")
version = project.version(1)
dataset = version.download("coco")

## 2. Install RF-DETR

In [ ]:
# Pinned: the version the Jetson's decode (dorna_vision/ai.py) is checked
# against — same ONNX inputs/outputs. Bump both together.
!pip install -q "rfdetr[train,onnx]==1.11.2"

## 3. Training Parameters

On the Jetson Orin Nano Super (FP16 TensorRT), `small` runs a frame in about 25 ms, plus ~2 ms per object for its outline.

In [ ]:
import os
import json
import pickle
from google.colab import files
from rfdetr import RFDETRSegNano, RFDETRSegSmall, RFDETRSegMedium, RFDETRSegLarge, RFDETRSegXLarge

# ============================================================
# EDIT THIS SECTION
# ============================================================
MODEL_SIZE = "small"       # "nano" | "small" | "medium" | "large" | "xlarge"
EPOCHS = 100
BATCH_SIZE = 8
GRAD_ACCUM = 2             # effective batch = BATCH_SIZE x GRAD_ACCUM (16 recommended)
LR = 1e-4
RESOLUTION = None          # None = the size's default; else a multiple of the size's patch grid
                           # (RF-DETR raises if not)

MODELS = {"nano": RFDETRSegNano, "small": RFDETRSegSmall, "medium": RFDETRSegMedium,
          "large": RFDETRSegLarge, "xlarge": RFDETRSegXLarge}
Model = MODELS[MODEL_SIZE]

project_name = project.name.lower().replace(" ", "_")

print(f"Model size:  {MODEL_SIZE}")
print(f"Epochs:      {EPOCHS}")
print(f"Batch size:  {BATCH_SIZE} x {GRAD_ACCUM}")
print(f"Resolution:  {RESOLUTION or 'model default'}")

## 4. Verify COCO Dataset Layout

Roboflow's COCO export is RF-DETR's native format — no conversion needed.
This cell sanity-checks the paths and reports what each split holds.

In [ ]:
# Roboflow's COCO export layout:
#   dataset.location/
#     train/_annotations.coco.json + images
#     valid/_annotations.coco.json + images
#     test/_annotations.coco.json  + images

DATA_ROOT = dataset.location

for split in ("train", "valid", "test"):
    ann_path = f"{DATA_ROOT}/{split}/_annotations.coco.json"
    if not os.path.exists(ann_path):
        print(f"  {split}: MISSING  ({ann_path})")
        continue
    with open(ann_path) as f:
        coco = json.load(f)
    print(f"  {split}: {len(coco.get('images', []))} images, {len(coco.get('annotations', []))} annotations")
    masks = sum(1 for a in coco.get("annotations", []) if a.get("segmentation"))
    print(f"           {masks} with masks")

with open(f"{DATA_ROOT}/train/_annotations.coco.json") as f:
    train_coco = json.load(f)

## 5. Train

In [ ]:
OUTPUT_DIR = "/content/output"

model = Model()
model.train(
    dataset_dir=DATA_ROOT,
    output_dir=OUTPUT_DIR,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    grad_accum_steps=GRAD_ACCUM,
    lr=LR,
    **({"resolution": RESOLUTION} if RESOLUTION else {}),
)

## 6. Export ONNX

The best checkpoint (by validation mAP) is exported. The check runs the ONNX
file on a validation image and compares its best box with RF-DETR's own
prediction — a bad export shows here, not on the bench.

In [ ]:
import cv2
import numpy as np
import onnxruntime as ort

best = Model(pretrain_weights=f"{OUTPUT_DIR}/checkpoint_best_total.pth")
onnx_path = best.export(output_dir="/content/export", verbose=False)

S = int(best.model_config.resolution)
val_dir = f"{DATA_ROOT}/valid"
sample = sorted(f for f in os.listdir(val_dir) if f.lower().endswith((".jpg", ".jpeg", ".png")))[0]
img = cv2.imread(f"{val_dir}/{sample}")
H, W = img.shape[:2]

# RF-DETR's predict() preprocessing: RGB, resize to S x S, ImageNet mean/std
rgb = cv2.resize(img, (S, S), interpolation=cv2.INTER_LINEAR)[:, :, ::-1].astype(np.float32) / 255
x = ((rgb - [0.485, 0.456, 0.406]) / [0.229, 0.224, 0.225]).transpose(2, 0, 1)[None].astype(np.float32)
session = ort.InferenceSession(onnx_path)
out = dict(zip([o.name for o in session.get_outputs()], session.run(None, {"input": x})))
q = (1 / (1 + np.exp(-out["labels"][0]))).max(1).argmax()
cx, cy, w, h = out["dets"][0][q]
onnx_box = np.array([(cx - w / 2) * W, (cy - h / 2) * H, (cx + w / 2) * W, (cy + h / 2) * H])

pred = best.predict(img[:, :, ::-1].copy(), threshold=0.0)
boxes = pred.xyxy if hasattr(pred, "xyxy") else pred.data["xyxy"]
scores = pred.confidence if hasattr(pred, "xyxy") else pred.detection_confidence
print("ONNX best box:   ", np.round(onnx_box, 1))
print("RF-DETR best box:", np.round(boxes[int(np.argmax(scores))], 1))

## 7. Save & Download Pickle

In [ ]:
import onnx
from rfdetr.assets.coco_classes import COCO_CLASSES

COCO_CLASS_NAMES = list(COCO_CLASSES.values())


def slot_names(model, num_slots):
    """The model's logit slots named as RF-DETR's predict() names them —
    a pretrained model's sparse COCO ids, a keypoint model's background
    slot, else slot i = class_names[i]. None marks a slot it never reports."""
    names = list(model.class_names)
    keypoints_per_class = list(getattr(model.model.args, "num_keypoints_per_class", None) or [])
    slots = [None] * num_slots
    if num_slots > len(names) and names == COCO_CLASS_NAMES:
        for i, coco_id in enumerate(COCO_CLASSES):
            if i < len(names):
                slots[coco_id] = names[i]
    elif keypoints_per_class and keypoints_per_class[0] == 0:
        foreground = [slot for slot, k in enumerate(keypoints_per_class) if k > 0]
        for i, slot in enumerate(foreground[:len(names)]):
            slots[slot] = names[i]
    else:
        for i, name in enumerate(names[:num_slots]):
            slots[i] = name
    return slots


def rfdetr_model_dict(model, onnx_path, model_type, variant, colors, keypoint_names=None, skeleton=None):
    """The pickle the Jetson's OD / SEG / KP load (dorna_vision/ai.py)."""
    with open(onnx_path, "rb") as f:
        onnx_data = f.read()
    graph = onnx.load_from_string(onnx_data)
    num_slots = next(o for o in graph.graph.output if o.name == "labels").type.tensor_type.shape.dim[2].dim_value
    model_dict = {
        "onnx": onnx_data,
        "cls": slot_names(model, num_slots),
        "colors": colors,
        "meta": {
            "type": model_type,
            "arch": "rf-detr",
            "variant": variant,
            "image_size": int(model.model_config.resolution),
            "precision": "fp16",   # the Jetson's TensorRT engine; RF-DETR's activations fit FP16
            # the engine takes the camera's uint8 BGR image; RF-DETR's graph wants RGB, ImageNet mean/std
            "input": {"format": "uint8-bgr", "channels": "rgb",
                      "mean": [0.485, 0.456, 0.406], "std": [0.229, 0.224, 0.225]},
        },
    }
    if model_type == "kp":
        model_dict["meta"]["num_keypoints_per_class"] = list(model.model.args.num_keypoints_per_class)
        model_dict["meta"]["trace_alpha"] = float(model.model.postprocess.trace_alpha)
        model_dict["keypoint_names"] = keypoint_names
        model_dict["skeleton"] = skeleton
    return model_dict

In [ ]:
try:
    colors = project.colors or {}
except Exception:
    colors = {}

model_dict = rfdetr_model_dict(best, onnx_path, "seg", MODEL_SIZE, colors)

pickle_path = f"/content/{project_name}.pkl"
with open(pickle_path, "wb") as f:
    pickle.dump(model_dict, f)

print(f"Saved: {pickle_path}")
print(f"Total size: {os.path.getsize(pickle_path) / 1024 / 1024:.1f} MB")
print(f"Classes: {[c for c in model_dict['cls'] if c is not None]}")
print(f"Image size: {model_dict['meta']['image_size']}")

files.download(pickle_path)